In [ ]:
import numpy as np
import h5py
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib as mpl
from matplotlib.ticker import AutoLocator, AutoMinorLocator, LogLocator
import glob
import cv2
import os
import re

In [ ]:
data_directory = '/pscratch/sd/b/bigqke/bigqke_scratch/emu_methods_paper_simulations/bigqke_per/shadow_test/shadow_test'
# data_directory = '/pscratch/sd/b/bigqke/bigqke_scratch/emu_methods_paper_simulations/bigqke_per/shadow_test/shadow_test_ndir_590'
# data_directory = '/pscratch/sd/b/bigqke/bigqke_scratch/emu_methods_paper_simulations/bigqke_per/shadow_test/shadow_test_ndir_846'
# data_directory = '/pscratch/sd/b/bigqke/bigqke_scratch/emu_methods_paper_simulations/bigqke_per/shadow_test/shadow_test_ndir_1158'
data_directory = '/pscratch/sd/b/bigqke/bigqke_scratch/emu_methods_paper_simulations/bigqke_per/shadow_test/shadow_test_ncell_192_64_64_dts_3.659e-07_ndir_2348_stored'
R = 1170000.0 # cm
r_sphere_radius = R # cm
r_sphere_center_cm = ( 1425937.5, 1425937.5, 1425937.5 ) # cm
r_sphere_radius_index = ( 32.0, 32.0, 32.0 ) # cells
r_shadowing_sphere_radius = 1.5*R/4.5 # cm

r_shadowing_sphere_center_cm = ( 6120562.5, 1425937.5, 1425937.5 ) # cm
r_shadowing_sphere_radius_index = ( 139.0, 32.0, 32.0 ) # cells

r_shadowing_sphere_center_cm = ( 4935937.5, 1425937.5, 1425937.5 )
r_shadowing_sphere_radius_index = ( 112.0, 32.0, 32.0 )

mesh_files = glob.glob(os.path.join(data_directory, 'mesh_plt*.h5'))
mesh_files_nums = [filenames.split('mesh_plt')[1].split('.h5')[0] for filenames in mesh_files]
mesh_files_sorted = [f for _, f in sorted(zip(mesh_files_nums, mesh_files))]
mesh_files_sorted = np.array(mesh_files_sorted)
nfiles = len(mesh_files_sorted)
print(f'Found {len(mesh_files_sorted)} mesh files.')

In [ ]:
rho_ye_T_h5py = h5py.File(data_directory+'/rho_Ye_T.hdf5', 'r')

# number of cells
Nx = np.array(rho_ye_T_h5py['/ncellsx'])
Ny = np.array(rho_ye_T_h5py['/ncellsy'])
Nz = np.array(rho_ye_T_h5py['/ncellsz'])
print(f'Nx = {Nx}, Ny = {Ny}, Nz = {Nz}')

# cell size
dx = ( np.array(rho_ye_T_h5py['/xmax']) - np.array(rho_ye_T_h5py['/xmin']) ) / np.array(rho_ye_T_h5py['/ncellsx']) # cm
dy = ( np.array(rho_ye_T_h5py['/ymax']) - np.array(rho_ye_T_h5py['/ymin']) ) / np.array(rho_ye_T_h5py['/ncellsy']) # cm
dz = ( np.array(rho_ye_T_h5py['/zmax']) - np.array(rho_ye_T_h5py['/zmin']) ) / np.array(rho_ye_T_h5py['/ncellsz']) # cm
print(f'dx = {dx} cm, dy = {dy} cm, dz = {dz} cm')

# cell centers
xc = np.linspace(dx / 2, dx * (Nx - 0.5), Nx) # cm
yc = np.linspace(dy / 2, dy * (Ny - 0.5), Ny) # cm
zc = np.linspace(dz / 2, dz * (Nz - 0.5), Nz) # cm

# cell centers mesh
Xc, Yc, Zc = np.meshgrid(xc, yc, zc, indexing='ij') # cm

bh_radius = 5.43e+05 # cm
bh_center_x = 48.0e+5 # cm
bh_center_y = 48.0e+5 # cm
bh_center_z = 16.0e+5 # cm

distance_from_bh = np.sqrt( (Xc - bh_center_x)**2 + (Yc - bh_center_y)**2 + (Zc - bh_center_z)**2 )
# mask_bh = (distance_from_bh < bh_radius + 1*np.max([dx,dy,dz]))
mask_bh = (distance_from_bh < bh_radius)
mask_x = (Xc < dx) | (Xc > (Nx - 1) * dx)
mask_y = (Yc < dy) | (Yc > (Ny - 1) * dy)
mask_z = (Zc < dz) | (Zc > (Nz - 1) * dz)
mask = mask_bh | mask_x | mask_y | mask_z

rho_ye_T_h5py.close()

In [ ]:
# Font settings
mpl.rcParams['font.size'] = 22
mpl.rcParams['font.family'] = 'serif'
mpl.rc('text', usetex=False)

# Tick settings
mpl.rcParams['xtick.major.width'] = 2
mpl.rcParams['xtick.major.pad'] = 8
mpl.rcParams['xtick.minor.size'] = 4

mpl.rcParams['xtick.minor.width'] = 2
mpl.rcParams['ytick.major.size'] = 7
mpl.rcParams['ytick.major.width'] = 2
mpl.rcParams['ytick.minor.size'] = 4
mpl.rcParams['ytick.minor.width'] = 2

# Axis linewidth
mpl.rcParams['axes.linewidth'] = 2

# Tick direction and enabling ticks on all sides
mpl.rcParams['xtick.direction'] = 'in'
mpl.rcParams['ytick.direction'] = 'in'
mpl.rcParams['xtick.top'] = True
mpl.rcParams['ytick.right'] = True

# Function to apply custom tick locators and other settings to an Axes object
def apply_custom_settings(ax, log_scale_y=False):

    if log_scale_y:
        # Use LogLocator for the y-axis if it's in log scale
        ax.set_yscale('log')
        ax.yaxis.set_major_locator(LogLocator(base=10.0))
        ax.yaxis.set_minor_locator(LogLocator(base=10.0, subs='auto', numticks=100))
    else:
        # Use AutoLocator for regular scales
        ax.yaxis.set_major_locator(AutoLocator())
        ax.yaxis.set_minor_locator(AutoMinorLocator())
    
    # Apply the AutoLocator for the x-axis
    ax.xaxis.set_major_locator(AutoLocator())
    ax.xaxis.set_minor_locator(AutoMinorLocator())

def plot_color_map(x, y, z, bh_r, bh_x, bh_y, min_cb, max_cb, x_label, y_label, title, cbar_label, colormap, filename, doshow=True, dosave=True):

    fig, ax = plt.subplots(figsize=(12, 8))

    # Plot pcolormesh
    # Set non-data points (NaN) to black in the colormap
    cmap = plt.get_cmap(colormap).copy()
    cmap.set_bad(color='black')

    # Plot pcolormesh with masked array to handle NaNs
    c = ax.pcolormesh(x, y, z, shading='auto', cmap=cmap, vmin=min_cb, vmax=max_cb)
    
    # Add contour lines
    # contour = ax.contour(x, y, z, colors='black', linewidths=1.5, levels=3)
    # contour = ax.contour(x, y, z, colors='black', linewidths=1.5, levels=[32.0, 32.2, 32.4])
    # ax.clabel(contour, inline=True, fontsize=15, fmt='%1.1f')

    # Plot settings
    ax.set_xlabel(x_label)
    ax.set_ylabel(y_label)
    # ax.set_title(title+'\nmin: {:.2e}\nmax: {:.2e}'.format(np.nanmin(z), np.nanmax(z)))
    ax.set_title(title)

    linescolors = 'black'
    linesalpha = 0.5

    # Emitter and absorber spheres. Centers and radii are stored in cm;
    # the plot axes are in km. Both spheres have y = z, so (x, y) is also
    # the correct (x, z) center on the xz slices.
    emitter_circle = plt.Circle(
        (r_sphere_center_cm[0] / 1e5, r_sphere_center_cm[1] / 1e5),
        radius=r_sphere_radius / 1e5,
        edgecolor=linescolors,
        facecolor='none',
        linewidth=2,
        label='emitter',
        zorder=5,
        alpha=linesalpha,
    )
    absorber_circle = plt.Circle(
        (r_shadowing_sphere_center_cm[0] / 1e5, r_shadowing_sphere_center_cm[1] / 1e5),
        radius=r_shadowing_sphere_radius / 1e5,
        edgecolor=linescolors,
        facecolor='none',
        linewidth=2,
        label='absorber',
        zorder=5,
        alpha=linesalpha,
    )
    ax.add_patch(emitter_circle)
    ax.add_patch(absorber_circle)
    # ax.legend(loc='upper right')

    # External tangents: one contact point on each sphere, extended to +x.
    emitter_center_km = np.array([
        r_sphere_center_cm[0] / 1e5,
        r_sphere_center_cm[1] / 1e5,
    ])
    absorber_center_km = np.array([
        r_shadowing_sphere_center_cm[0] / 1e5,
        r_shadowing_sphere_center_cm[1] / 1e5,
    ])
    emitter_radius_km = r_sphere_radius / 1e5
    absorber_radius_km = r_shadowing_sphere_radius / 1e5
    center_separation = absorber_center_km - emitter_center_km
    separation = np.linalg.norm(center_separation)
    line_angle = np.arctan2(center_separation[1], center_separation[0])
    tangent_tilt = np.arcsin((emitter_radius_km - absorber_radius_km) / separation)
    x_domain_max = np.nanmax(x)
    for side in (+1, -1):  # upper, then lower
        radius_angle = line_angle + side * (np.pi / 2 - tangent_tilt)
        radius_direction = np.array([np.cos(radius_angle), np.sin(radius_angle)])
        emitter_touch = emitter_center_km + emitter_radius_km * radius_direction
        absorber_touch = absorber_center_km + absorber_radius_km * radius_direction
        tangent_direction = absorber_touch - emitter_touch
        extension = (x_domain_max - emitter_touch[0]) / tangent_direction[0]
        tangent_end = emitter_touch + extension * tangent_direction
        ax.plot(
            [emitter_touch[0], tangent_end[0]],
            [emitter_touch[1], tangent_end[1]],
            color=linescolors,
            linewidth=2,
            zorder=6,
            alpha=linesalpha,
        )

    # Internal tangents: opposite contact sides, extended to +x.
    internal_tilt = np.arcsin((emitter_radius_km + absorber_radius_km) / separation)
    for side in (+1, -1):
        radius_angle = line_angle + side * (np.pi / 2 - internal_tilt)
        radius_direction = np.array([np.cos(radius_angle), np.sin(radius_angle)])
        emitter_touch = emitter_center_km + emitter_radius_km * radius_direction
        absorber_touch = absorber_center_km - absorber_radius_km * radius_direction
        tangent_direction = absorber_touch - emitter_touch
        extension = (x_domain_max - emitter_touch[0]) / tangent_direction[0]
        tangent_end = emitter_touch + extension * tangent_direction
        ax.plot(
            [emitter_touch[0], tangent_end[0]],
            [emitter_touch[1], tangent_end[1]],
            color=linescolors,
            linewidth=2,
            zorder=6,
            alpha=linesalpha,
        )

    # Add color bar
    cbar = fig.colorbar(c, ax=ax, label=cbar_label)
    cbar.ax.yaxis.set_minor_locator(AutoMinorLocator())

    apply_custom_settings(ax, False)

    # Lock the view to the mesh extent. x and y are 2D, so the limits have
    # to be scalars. Set them after the circles, tangents, and equal aspect,
    # and turn autoscaling off so those artists cannot expand the frame.
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlim(float(np.nanmin(x)), float(np.nanmax(x)), auto=False)
    ax.set_ylim(float(np.nanmin(y)), float(np.nanmax(y)), auto=False)

    # Save figure
    if dosave:
        fig.savefig(filename, format='png', bbox_inches='tight')

    # Display figure
    if doshow:
        plt.show()
    # display(fig)
    
    # Close figure
    plt.close(fig)

In [ ]:
x_slice_indx = 96 # cell number
y_slice_indx = 32 # cell number
z_slice_indx = 32 # cell number
time_indx    = -1 # last time step

# for time_indx in [8,12,16,20,24,28,32,36,40]:
# for time_indx in [1,2,3,4,5,6,7,8,9,10]:

for filepath in [mesh_files_sorted[-1]]:
# for filepath in mesh_files_sorted:

    print(f'Processing file: {filepath}')

    alldata_h5 = h5py.File(filepath, 'r')
    alldata_dict = {key: np.array(alldata_h5[key]) for key in alldata_h5.keys()}
    alldata_h5.close()
    alldata = alldata_dict.copy()
    print("Keys in alldata_dict:", list(alldata.keys()))

    # if time_indx==0:
    if True:
        doshow = True
    else:
        doshow = False

    time_ms = np.array(alldata['t(s)']) * 1.0e3 # ms

    # global min: -13.821983599860665 global max: 24.104102117542855
    max_cbar = 24.104102117542855
    min_cbar = 24.104102117542855 - 10
    # min_cbar = 0
    # max_cbar = 2e22
    # min_cbar = 1e15
    # max_cbar = None
    # min_cbar = None

    ###############################################
    # Neutrino diagonal terms
    ###############################################

    # # Electron neutrino number density
    zup = np.where(mask[1:-1, 1:-1, z_slice_indx], np.nan, np.array(alldata['N00_Re(1|ccm)'])[1:-1, 1:-1, z_slice_indx])
    zlow = np.where(mask[1:-1, y_slice_indx, 1:-1], np.nan, np.array(alldata['N00_Re(1|ccm)'])[1:-1, y_slice_indx, 1:-1])
    zup = np.log10(zup)
    zlow = np.log10(zlow)
    # zup = zup
    # zlow = zlow
    print("global min:", np.nanmin([np.nanmin(zup), np.nanmin(zlow)]), "global max:", np.nanmax([np.nanmax(zup), np.nanmax(zlow)]))
    colorbarlabel = fr'$\log ( \, n_{{\nu_{{e}}}}\,[\mathrm{{cm}}^{{-3}}]\,)$'
    filename=f'./plots/{filepath.split('mesh_plt')[1].split('.h5')[0]}_nee'
    mincbar=min_cbar
    maxcbar=max_cbar
    # mincbar=None
    # maxcbar=None
    
    plot_color_map(
        x=Xc[1:-1, 1:-1, z_slice_indx] / 1e5,
        y=Yc[1:-1, 1:-1, z_slice_indx] / 1e5,
        z=zup,
        bh_r=6.8,
        bh_x=48.0,
        bh_y=48.0,
        min_cb=mincbar,
        max_cb=maxcbar,
        x_label=r'$x \, (\mathrm{km})$',
        y_label=r'$y \, (\mathrm{km})$',
        title=fr'$z={Zc[0, 0, z_slice_indx] / 1e5:.1f}\,\mathrm{{km}}$, $t = {time_ms:.5f}\,\mathrm{{ms}}$',
        cbar_label=colorbarlabel,
        colormap='rainbow',
        filename=f'{filename}_xy.png',
        doshow=doshow,
        dosave=True,
    )
    plot_color_map(
        x=Xc[1:-1, y_slice_indx, 1:-1] / 1e5,
        y=Zc[1:-1, y_slice_indx, 1:-1] / 1e5,
        z=zlow,
        bh_r=6.8,
        bh_x=48.0,
        bh_y=16.0,
        min_cb=mincbar,
        max_cb=maxcbar,
        x_label=r'$x \, (\mathrm{km})$',
        y_label=r'$z \, (\mathrm{km})$',
        title=fr'$y={Yc[0, y_slice_indx, 0] / 1e5:.1f}\,\mathrm{{km}}$, $t = {time_ms:.5f}\,\mathrm{{ms}}$',
        cbar_label=colorbarlabel,
        colormap='rainbow',
        filename=f'{filename}_xz.png',
        doshow=doshow,
        dosave=True,
    )